# K Nearest Neighbor (KNN)
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
KNN models can be used for both classification and prediction.  They are built on a simple premise and can perform well when you are just looking for a local prediction, not an overall summary of the patterns in the data.  


# Environment Setup

In [ ]:
# import modules

import pandas as pd # for data viz and wrangling
import numpy as np # for 'numeric python'
import matplotlib.pyplot as plt # for data viz (more complex than pylab)
import seaborn as sns
from pylab import * # for data viz (import * means 'import all of the functions')

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn import metrics
from scipy import stats
import statsmodels.api as sm


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

#Load the Data - Universal Bank

KNN can work with both data types for the target variable.  First Let's use if for classification.  The Universal Bank dataset can be used to predict if a customer will accept an offer of a personal loan.  Each row represents a bank customer and the target variable is called Personal Loan.

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 8/UniversalBank.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 5000 rows and 14 columns
df.shape

In [ ]:
# Preview
print(df.head())

In [ ]:
# list the columns with the data types
print(df.info())

Looks like a mix of categorical and continuous predictors.  We should  change our target variable to 1s and 0s.  And let's make dummy variables for Education.  We should drop Zip Code because it has too many categories.  

In [ ]:
# Convert 'Personal Loan' to numerical (Yes=1, No=0) using map
df['Personal Loan'] = df['Personal Loan'].map({'Yes': 1, 'No': 0})

# Display the first few rows and unique values to see the changes in the target variable
print(df.head())
print("\nUnique values after conversion:")
print(df['Personal Loan'].unique())

In [ ]:
# Look at counts of 0 and 1 values for Personal Loan to check our work
personal_loan_counts = df['Personal Loan'].value_counts()
print("Counts of Personal Loan (0s and 1s):")
print(personal_loan_counts)

In [ ]:
# Drop the 'ZIP Code' column
df = df.drop('ZIP Code', axis=1)

In [ ]:
# Create dummy variables for 'Education'
df = pd.get_dummies(df, columns=['Education'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Now let's partition the data to get it ready for modeling. We define Personal Loan as the target variable and the rest of the columns as our predictor variables.  We will do a 50/30/20 split.  In the code we first separate out the 20% for test, then we split the remaining portion into training and validation.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Personal Loan'
# Define features by dropping the target variable and the 'ID' column
features = df.drop([target, 'ID'], axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Combine the features and target for plotting
train_data = X_train.copy()
train_data['Personal Loan'] = y_train

val_data = X_val.copy()
val_data['Personal Loan'] = y_val

test_data = X_test.copy()
test_data['Personal Loan'] = y_test

# Plot the distribution of 'Personal Loan' in each partition
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

sns.countplot(data=train_data, x='Personal Loan', ax=axs[0])
axs[0].set_title('Training Set - Personal Loan Distribution')
axs[0].set_xlabel('Personal Loan')
axs[0].set_ylabel('Count')
axs[0].set_xticks([0, 1])
axs[0].set_xticklabels(['No', 'Yes'])

sns.countplot(data=val_data, x='Personal Loan', ax=axs[1])
axs[1].set_title('Validation Set - Personal Loan Distribution')
axs[1].set_xlabel('Personal Loan')
axs[1].set_ylabel('Count')
axs[1].set_xticks([0, 1])
axs[1].set_xticklabels(['No', 'Yes'])


sns.countplot(data=test_data, x='Personal Loan', ax=axs[2])
axs[2].set_title('Test Set - Personal Loan Distribution')
axs[2].set_xlabel('Personal Loan')
axs[2].set_ylabel('Count')
axs[2].set_xticks([0, 1])
axs[2].set_xticklabels(['No', 'Yes'])

plt.tight_layout()
plt.show()

# Print counts and percentages for each partition
print("Training set Personal Loan counts and percentages:")
train_counts = train_data['Personal Loan'].value_counts()
train_percentages = train_data['Personal Loan'].value_counts(normalize=True) * 100
print(train_counts)
print(train_percentages.round(2)) # Round percentages to 2 decimal places


print("\nValidation set Personal Loan counts and percentages:")
val_counts = val_data['Personal Loan'].value_counts()
val_percentages = val_data['Personal Loan'].value_counts(normalize=True) * 100
print(val_counts)
print(val_percentages.round(2)) # Round percentages to 2 decimal places


print("\nTest set Personal Loan counts and percentages:")
test_counts = test_data['Personal Loan'].value_counts()
test_percentages = test_data['Personal Loan'].value_counts(normalize=True) * 100
print(test_counts)
print(test_percentages.round(2)) # Round percentages to 2 decimal places

#KNN for Classification

We don't know what number of neighbors is best to consider so we will try a range of values.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define a range of n_neighbors values to try
n_neighbors_range = range(1, 21) # Example range from 1 to 20

# Create lists to store results
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Iterate through the range of n_neighbors values
for n in n_neighbors_range:
    # Initialize the KNN classifier with the current n_neighbors
    knn_classifier = KNeighborsClassifier(n_neighbors=n)

    # Train the model
    knn_classifier.fit(X_train, y_train)

    # Make predictions on the validation set
    y_pred_val = knn_classifier.predict(X_val)

    # Evaluate the model on the validation set
    accuracy = accuracy_score(y_val, y_pred_val)
    precision = precision_score(y_val, y_pred_val)
    recall = recall_score(y_val, y_pred_val)
    f1 = f1_score(y_val, y_pred_val)

    # Store the scores
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

# Now you have the performance metrics for each n_neighbors in the lists
# You can analyze these lists to find the best n_neighbors based on your preferred metric

Let's look at our common performance measures across the different numbers of neighbors.

In [ ]:
# Plot the performance metrics for each n_neighbors
plt.figure(figsize=(12, 8))

plt.plot(n_neighbors_range, accuracy_scores, label='Accuracy')
plt.plot(n_neighbors_range, precision_scores, label='Precision')
plt.plot(n_neighbors_range, recall_scores, label='Recall')
plt.plot(n_neighbors_range, f1_scores, label='F1-score')

plt.xlabel('Number of Neighbors (n_neighbors)')
plt.ylabel('Score')
plt.title('KNN Performance on Validation Set for Different n_neighbors')
plt.xticks(n_neighbors_range)
plt.legend()
plt.grid(True)
plt.show()

Based on this plot, K=5 looks good.  It has the highest F1 Score and the highest Recall.  The Precision is nearly at the highest too.  

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the KNN classifier with n_neighbors=5
knn_classifier_5 = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_classifier_5.fit(X_train, y_train)

# Make predictions on the validation set
y_pred_val_5 = knn_classifier_5.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate on the training set
y_pred_train_5 = knn_classifier_5.predict(X_train)
accuracy_train = accuracy_score(y_train, y_pred_train_5)
precision_train = precision_score(y_train, y_pred_train_5)
recall_train = recall_score(y_train, y_pred_train_5)
f1_train = f1_score(y_train, y_pred_train_5)

print("Performance metrics for Training Set (n_neighbors=5):")
print(f"Accuracy: {accuracy_train:.4f}")
print(f"Precision: {precision_train:.4f}")
print(f"Recall: {recall_train:.4f}")
print(f"F1-score: {f1_train:.4f}")

# Evaluate on the validation set (predictions already made)
accuracy_val = accuracy_score(y_val, y_pred_val_5)
precision_val = precision_score(y_val, y_pred_val_5)
recall_val = recall_score(y_val, y_pred_val_5)
f1_val = f1_score(y_val, y_pred_val_5)

print("\nPerformance metrics for Validation Set (n_neighbors=5):")
print(f"Accuracy: {accuracy_val:.4f}")
print(f"Precision: {precision_val:.4f}")
print(f"Recall: {recall_val:.4f}")
print(f"F1-score: {f1_val:.4f}")

# Evaluate on the test set
y_pred_test_5 = knn_classifier_5.predict(X_test)
accuracy_test = accuracy_score(y_test, y_pred_test_5)
precision_test = precision_score(y_test, y_pred_test_5)
recall_test = recall_score(y_test, y_pred_test_5)
f1_test = f1_score(y_test, y_pred_test_5)

print("\nPerformance metrics for Test Set (n_neighbors=5):")
print(f"Accuracy: {accuracy_test:.4f}")
print(f"Precision: {precision_test:.4f}")
print(f"Recall: {recall_test:.4f}")
print(f"F1-score: {f1_test:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Create a figure with three subplots side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5)) # 1 row, 3 columns

# Confusion matrix for training set
sns.heatmap(confusion_matrix(y_train, y_pred_train_5), annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix - Training Set (n_neighbors=5)')

# Confusion matrix for validation set
sns.heatmap(confusion_matrix(y_val, y_pred_val_5), annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'], ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix - Validation Set (n_neighbors=5)')

# Confusion matrix for test set
sns.heatmap(confusion_matrix(y_test, y_pred_test_5), annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'], ax=axes[2])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix - Test Set (n_neighbors=5)')

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

For the Universal bank data, the KNN model didn't perform all that great compared to some other models that we've built with higher metrics.

#Load the Data - ToyotaCorolla1000

Now let's do our KNN model again, but this time with a continuous target.  

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 8/ToyotaCorolla1000.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 1000 rows and 10 columns
df.shape

In [ ]:
# Preview
print(df.head())

In [ ]:
# Create dummy variables for 'Fuel Type'
df = pd.get_dummies(df, columns=['Fuel Type'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Before we model, we need to partition the data.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Price'
features = df.drop(target, axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

#KNN for Prediction

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np # Import numpy for square root

# Define a range of n_neighbors values to try
n_neighbors_range = range(1, 21) # Example range from 1 to 20

# Create lists to store results
rmse_scores = [] # Change from mse_scores to rmse_scores
r2_scores = []

# Iterate through the range of n_neighbors values
for n in n_neighbors_range:
    # Initialize the KNN regressor with the current n_neighbors
    knn_regressor = KNeighborsRegressor(n_neighbors=n)

    # Train the model
    knn_regressor.fit(X_train, y_train)

    # Make predictions on the validation set
    y_pred_val = knn_regressor.predict(X_val)

    # Evaluate the model on the validation set using RMSE and R-squared
    mse = mean_squared_error(y_val, y_pred_val)
    rmse = np.sqrt(mse) # Calculate RMSE
    r2 = r2_score(y_val, y_pred_val)

    # Store the scores
    rmse_scores.append(rmse) # Store RMSE
    r2_scores.append(r2)

# Now you have the performance metrics (RMSE and R-squared) for each n_neighbors in the lists
# You can analyze these lists to find the best n_neighbors based on your preferred metric

Let's check out the performance.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np # Import numpy for square root

# Plot RMSE and R-squared for each n_neighbors
plt.figure(figsize=(12, 6))

# Plot RMSE
plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
plt.plot(n_neighbors_range, rmse_scores, marker='o') # Change to rmse_scores
plt.xlabel('Number of Neighbors (n_neighbors)')
plt.ylabel('Root Mean Squared Error (RMSE)') # Change ylabel
plt.title('KNN Regression Performance (RMSE) on Validation Set') # Change title
plt.xticks(n_neighbors_range)
plt.grid(True)

# Format y-axis to not use scientific notation
formatter = mticker.ScalarFormatter(useMathText=True)
formatter.set_powerlimits((-3, 5))
plt.gca().yaxis.set_major_formatter(formatter)


# Plot R-squared
plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
plt.plot(n_neighbors_range, r2_scores, marker='o', color='green')
plt.xlabel('Number of Neighbors (n_neighbors)')
plt.ylabel('R-squared')
plt.title('KNN Regression Performance (R-squared) on Validation Set')
plt.xticks(n_neighbors_range)
plt.grid(True)

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

K=18 looks good here!  It gives us the lowest RMSE and the highest R squared.  

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Initialize the KNN regressor with n_neighbors=18
knn_regressor_18 = KNeighborsRegressor(n_neighbors=18)

# Train the model on the training data
knn_regressor_18.fit(X_train, y_train)

# Make predictions on the validation and test sets
y_pred_val_18 = knn_regressor_18.predict(X_val)
y_pred_test_18 = knn_regressor_18.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Evaluate on the training set
y_pred_train_18 = knn_regressor_18.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train_18))
r2_train = r2_score(y_train, y_pred_train_18)

print("Performance metrics for Training Set (n_neighbors=18):")
print(f"RMSE: {rmse_train:.4f}")
print(f"R-squared: {r2_train:.4f}")

# Evaluate on the validation set
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val_18))
r2_val = r2_score(y_val, y_pred_val_18)

print("\nPerformance metrics for Validation Set (n_neighbors=18):")
print(f"RMSE: {rmse_val:.4f}")
print(f"R-squared: {r2_val:.4f}")

# Evaluate on the test set
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test_18))
r2_test = r2_score(y_test, y_pred_test_18)

print("\nPerformance metrics for Test Set (n_neighbors=18):")
print(f"RMSE: {rmse_test:.4f}")
print(f"R-squared: {r2_test:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure with three subplots side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot actual vs. predicted for training set
sns.scatterplot(x=y_train, y=y_pred_train_18, ax=axes[0])
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title('Actual vs. Predicted Price - Training Set (n_neighbors=18)')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], color='red', linestyle='--') # Add a diagonal line

# Plot actual vs. predicted for validation set
sns.scatterplot(x=y_val, y=y_pred_val_18, ax=axes[1])
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].set_title('Actual vs. Predicted Price - Validation Set (n_neighbors=18)')
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], color='red', linestyle='--') # Add a diagonal line


# Plot actual vs. predicted for test set
sns.scatterplot(x=y_test, y=y_pred_test_18, ax=axes[2])
axes[2].set_xlabel('Actual Price')
axes[2].set_ylabel('Predicted Price')
axes[2].set_title('Actual vs. Predicted Price - Test Set (n_neighbors=18)')
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--') # Add a diagonal line


plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

Again, this model isn't performing as well as some other models we've built to predict Price.  But it's a good model to try when your other models are under-performing.